# Price Statistics – Plots
stETH discount/premium, price volatility, and market depth (trading volume).

In [ ]:
import stata_setup
stata_setup.config('/home/yichen', 'mp')

In [ ]:
%%stata

clear all
set more off

global PROCESSED_DATA "../../processed_data"
global FIGURES        "../../lido-bank/figures"

In [ ]:
%%stata
* Install required community packages (safe to re-run; skips if already installed)
capture ssc install palettes, replace
capture ssc install colrspace, replace

In [ ]:
%%stata
****************************
* Figure: stETH Discount / Premium Time Series
****************************

import delimited using "$PROCESSED_DATA/steth_eth_daily.csv", varnames(1) clear
gen date2 = date(date, "YMD")
format date2 %td

replace discount_mean = discount_mean * 100

local v2 = date("2023-05-15", "YMD")

* Fake single-point series with matching style for legend entry only
gen v2_fake = discount_mean if date2 == `v2'

twoway ///
    (line discount_mean date2, lcolor(navy) lwidth(medthick)) ///
    (line v2_fake date2, lcolor(maroon) lwidth(thin) lpattern(dash) msymbol(none)) ///
    , ///
    yline(0, lcolor(black) lwidth(thin) lpattern(dash)) ///
    xline(`v2', lcolor(maroon) lwidth(thin) lpattern(dash)) ///
    xtitle("") ///
    ytitle("Discount / Premium (% vs ETH)") ///
    xlabel(, format(%tdMon-CCYY) angle(45)) ///
    legend(order(2 "Withdrawals enabled") pos(2) ring(0) size(medsmall)) ///
    graphregion(color(white)) ///
    plotregion(margin(zero))

graph export "$FIGURES/steth_discount_ts.pdf", replace

In [ ]:
%%stata
****************************
* Figure: stETH 7-Day Rolling Annualized Volatility
****************************

import delimited using "$PROCESSED_DATA/steth_eth_daily.csv", varnames(1) clear
gen date2 = date(date, "YMD")
format date2 %td

replace steth_vol_7d = steth_vol_7d * 100
drop if missing(steth_vol_7d)

local v2 = date("2023-05-15", "YMD")

* Fake single-point series with matching style for legend entry only
gen v2_fake = steth_vol_7d if date2 == `v2'

twoway ///
    (line steth_vol_7d date2, lcolor(navy) lwidth(medthick)) ///
    (line v2_fake date2, lcolor(maroon) lwidth(thin) lpattern(dash) msymbol(none)) ///
    , ///
    xline(`v2', lcolor(maroon) lwidth(thin) lpattern(dash)) ///
    xtitle("") ///
    ytitle("Annualized Volatility (%)") ///
    xlabel(, format(%tdMon-CCYY) angle(45)) ///
    legend(order(2 "Withdrawals enabled") pos(2) ring(0) size(medsmall)) ///
    graphregion(color(white)) ///
    plotregion(margin(zero))

graph export "$FIGURES/steth_vol_ts.pdf", replace

In [ ]:
%%stata
****************************
* Figure: stETH 24h Trading Volume
****************************

import delimited using "$PROCESSED_DATA/steth_eth_daily.csv", varnames(1) clear
gen date2 = date(date, "YMD")
format date2 %td

gen steth_vol_bn = steth_volume / 1e9

local v2 = date("2023-05-15", "YMD")

* Fake single-point series with matching style for legend entry only
gen v2_fake = steth_vol_bn if date2 == `v2'

twoway ///
    (line steth_vol_bn date2, lcolor(navy) lwidth(medthick)) ///
    (line v2_fake date2, lcolor(maroon) lwidth(thin) lpattern(dash) msymbol(none)) ///
    , ///
    xline(`v2', lcolor(maroon) lwidth(thin) lpattern(dash)) ///
    xtitle("") ///
    ytitle("24h Trading Volume (USD bn)") ///
    xlabel(, format(%tdMon-CCYY) angle(45)) ///
    legend(order(2 "Withdrawals enabled") pos(2) ring(0) size(medsmall)) ///
    graphregion(color(white)) ///
    plotregion(margin(zero))

graph export "$FIGURES/steth_volume_ts.pdf", replace